In [2]:
from langchain_openai import ChatOpenAI 
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model ="gpt-5-mini") 


### Tools

In [3]:
from langchain.tools import tool

In [4]:
@tool # Decorator to mark this function as a tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool when you need to answer questions about current events or general knowledge questions."""
   
    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)
    return response

In [5]:
tool_duckduckgo_search.invoke("What is the capital of India?")

"This is a list of locations which have served as capital cities in India. The current capital city is New Delhi, which replaced Calcutta in 1911. India Standard Time (IST) now 9 hours and 30 minutes ahead of New York.New Delhi is the capital of India. Latitude: 28.62. Longitude: 77.21. India's administrative divisions of States and Union Territories and their capitals. Fullscreen. Country: India. Long Name: Republic of India. Abbreviations: IN, IND. Capital: New Delhi. The capital of India is New Delhi, which was established in 1911 during British rule. It is a key governmental hub, housing important institutions and landmarks."

In [6]:
@tool # Decorator to mark this function as a tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about persons, places etc."""
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

In [7]:
tool_wikipedia_search.invoke("Who is the president of USA?")

'Page: List of presidents of the United States\nSummary: The president of the United States is the head of state and head of government of the United States, indirectly elected to a four-year term via the Electoral College. Under the U.S. Constitution, the officeholder leads the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.\nThe first president, George Washington, won a unanimous vote of the Electoral College. The incumbent president is Donald Trump, who assumed office on January 20, 2025. Since the office was established in 1789, 45 men have served in 47 presidencies. The discrepancy is due to the nonconsecutive terms of Grover Cleveland (counted as the 22nd and 24th president) and Trump (counted as the 45th and 47th president).\nThe presidency of William Henry Harrison, who died 31 days after taking office in 1841, was the shortest in American history. Franklin D. Roosevelt served the longest, over twelve years, before dyi

In [8]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [41]:
tool_arxiv_search.invoke("What are the latest research papers on deep learning?")

Published: 2023-01-03
Title: Deep Learning and Computational Physics (Lecture Notes)
Authors: Deep Ray, Orazio Pinti, Assad A. Oberai
Summary: These notes were compiled as lecture notes for a course developed and taught at the University of the Southern California. They should be accessible to a typical engineering graduate student with a strong background in Applied Mathematics.
  The main objective of these notes is to introduce a student who is familiar with concepts in linear algebra and partial differential equations to select topics in deep learning. These lecture notes exploit the strong connections between deep learning algorithms and the more conventional techniques of computational physics to achieve two goals. First, they use concepts from computational physics to develop an understanding of deep learning algorithms. Not surprisingly, many concepts in deep learning can be connected to similar concepts in computational physics, and one can utilize this connection to better un

In [9]:
@tool
def tool_personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Like name, age, occupation etc.
        Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    infos = [{
        "name": "John Doe",
        "age": 30,
        "Occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "Occupation": "Data Scientist"
    }]
    for info in infos:
        if query.lower() in info["name"].lower():
            return f"{info['name']} is a {info['age']} year old and works as a {info['Occupation']}."
    return "No information found."

In [43]:
tool_personal_info.invoke("John Doe")

'John Doe is a 30 year old and works as a Software Engineer.'

In [10]:
if os.path.exists("./chroma_db_semantic"):
    print("Chroma database already exists. Skipping creation.")

Chroma database already exists. Skipping creation.


In [11]:
@tool
def tool_rag_search(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSphere Organization's documentation. """
    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embed_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
    chroma_db_con = Chroma(persist_directory = "./chroma_db_semantic", embedding_function=embed_model)

    #1. Retrieve relevant documents from the Chroma database
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])

    return relevant_docs_content


In [12]:
tool_rag_search.invoke("What is NovaSphere Organization?")

C:\Users\hp\AppData\Local\Temp\ipykernel_8084\3744645833.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory = "./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

### Binding Tools

In [13]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag_search
    
]

In [14]:
llm_bind = llm.bind_tools(toolkit) #.bind_tools() is function that binds the tools to the llm. It takes a list of tools as input and returns a new llm that has access to those tools.

In [16]:
llm_bind.invoke("What is the age of John Doe? Make tool calls if necessary to answer the question.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 328, 'total_tokens': 354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWobdhgIA7FbE5s0DMc2jpaWciGcQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dac5e-e97f-7f92-bc1a-5673c0b59cab-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_mD4sA296pwNoVhrlPQ12OxSU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 328, 'output_tokens': 26, 'total_tokens': 354, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### ReAct Agent

In [15]:
from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [18]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary to answer the question."}]}
)

{'messages': [HumanMessage(content='What is the age of John Doe? Make tool calls if necessary to answer the question.', additional_kwargs={}, response_metadata={}, id='31a5c958-0d55-40d8-a475-5e9144a01e71'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 328, 'total_tokens': 482, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWocU2TKtpLuBAVZIu1SlNAOq3bFm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dac5f-b96c-7922-b571-baf67fb21c43-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_fz1R4Qvb0d0q4Dz0lUJsvaTG', 'type': 'tool_call'}], invalid_to

In [16]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded? "}]}
)

{'messages': [HumanMessage(content='When was NovaSphere Organization founded? ', additional_kwargs={}, response_metadata={}, id='08e2dbe3-d132-434a-9d9e-945974f4a1bf'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 318, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWpY3e01VnlC7gKw3uFpk8dAjUK0S', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dac96-3258-73f1-9f3f-a93a098a2bc3-0', tool_calls=[{'name': 'tool_rag_search', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_c1d1lmM3AC8dhWbftSAp0Oin', 'type': 'tool_call'}], invalid_tool_calls=[],